In [92]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import copy

In [4]:
df = pd.read_csv('df_clean.csv',index_col=0)
df

,!age,!gender_f,!gender_m,!gender_x,!height,!weight,!stress,!sleep,!caffeine,!calories,...,!carbs,!fat,!steps,!work_carb,!work_na,!work_strn,!work_yoga,!workintensity,!temp,#y#
0,0.23,1,0,0,0.493553,0.335999,1.0,0.584704,0.325501,0.667087,...,0.656667,0.583750,0.2288,0,0,1,0,0.6,0.535223,0.317158
1,0.43,1,0,0,0.544249,0.304904,0.1,0.610019,0.327326,0.634762,...,0.624792,0.555625,0.1552,0,1,0,0,0.0,0.115220,0.211877
2,0.47,0,1,0,0.432463,0.326694,0.3,0.603220,0.183322,0.406449,...,0.400000,0.355625,0.1504,0,0,0,1,1.0,0.128169,0.122390
3,0.42,1,0,0,0.449973,0.059731,0.7,0.421725,0.651229,0.630347,...,0.620417,0.551250,0.2412,1,0,0,0,1.0,0.171218,0.308449
4,0.54,0,1,0,0.476962,0.778482,0.5,0.822233,0.755467,0.484897,...,0.477292,0.424375,0.1696,0,1,0,0,0.0,0.528018,0.120221
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54995,0.31,1,0,0,0.529898,0.694361,0.6,0.482596,0.541485,0.540201,...,0.531667,0.472500,0.1340,0,1,0,0,0.0,0.499291,0.175921
54996,0.37,0,1,0,0.596888,0.562316,0.9,0.640981,0.696749,0.439391,...,0.432500,0.384375,0.1396,0,1,0,0,0.0,0.530780,0.175562
54997,0.19,1,0,0,0.473021,0.159519,0.3,0.695038,0.117568,0.499459,...,0.562292,0.500000,0.1236,0,1,0,0,0.0,0.612946,0.113811
54998,0.20,0,1,0,0.584018,0.249785,1.0,0.486085,0.869247,0.573279,...,0.564375,0.501875,0.1188,0,1,0,0,0.0,0.167526,0.340965


In [69]:
class LWDataset(Dataset):
    
    def __init__(self, df):
        self.x = torch.tensor(df[df.columns[df.columns.str.contains('!')]].values,dtype=torch.float32)
        self.y = torch.tensor(df['#y#'].values,dtype=torch.float32).unsqueeze(dim=1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [70]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_train, test_size=0.25, random_state=42)

dataset_train = LWDataset(df_train)
dataset_test = LWDataset(df_test)
dataset_val = LWDataset(df_val)

In [51]:
class MLP(nn.Module):
 
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 48)
        self.fc2 = nn.Linear(48, 12)
        self.fc3 = nn.Linear(12, 3)
        self.fc4 = nn.Linear(3, 1)
 
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        y_pred = self.fc4(x)
        return y_pred

In [72]:
for epoch in range(4):
    print(epoch)

0
1
2
3


In [73]:
np.mean([1,2,3])

2.0

In [111]:
def train(model,dataset_train,dataset_val):
    lr = 0.001
    n_epochs = 48
    warm_up_epoch_num = 5
    loss_tolerance_epoch_num = 5
    
    dataloader_train = DataLoader(dataset_train,batch_size=128, shuffle=True)
    
    model.train()
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    lowest_loss_epoch = {'loss_val':float("inf"),'epoch':0}
    best_score = -float('inf')
    best_model = copy.deepcopy(model.state_dict())
    
    log = {'loss_t':[],'loss_v':[]}
    
    for epoch in range(n_epochs):
        # train
        loss_train = []
        for x, y_true in dataloader_train:
            optimizer.zero_grad()
            y_pred = model(x)
            loss = criterion(y_pred, y_true)
            loss.backward()
            optimizer.step()
            loss_train.append(loss.item())
        loss_train = np.mean(loss_train)
        log['loss_t'].append(loss_train)
        # validate
        x = dataset_val.x
        y_true = dataset_val.y
        y_pred = model(x)
        loss_val = criterion(y_pred, y_true).item()
        log['loss_v'].append(loss_val)
        
        if loss_val < lowest_loss_epoch['loss_val']:
            lowest_loss_epoch['epoch'] = epoch
            lowest_loss_epoch['loss_val'] = loss_val
            print('★',end='')
            best_model = copy.deepcopy(model.state_dict())
        elif (epoch-lowest_loss_epoch['epoch']) >= loss_tolerance_epoch_num:
            lowest_loss_epoch['epoch'] = epoch
            print('○',end='')
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 0.5
        else: print(' ',end='')
        
        print('epoch ',epoch,
              f'\tloss_t: {loss_train:.6f}',
              f'\tloss_v: {loss_val:.6f}',sep='')
        
    return best_model,log

In [121]:
model = MLP()
best_model_state,log = train(model,dataset_train,dataset_val)

★epoch 0	loss_t: 0.002517	loss_v: 0.000227
★epoch 1	loss_t: 0.000145	loss_v: 0.000093
★epoch 2	loss_t: 0.000084	loss_v: 0.000071
★epoch 3	loss_t: 0.000065	loss_v: 0.000054
★epoch 4	loss_t: 0.000052	loss_v: 0.000040
★epoch 5	loss_t: 0.000034	loss_v: 0.000025
★epoch 6	loss_t: 0.000023	loss_v: 0.000018
★epoch 7	loss_t: 0.000018	loss_v: 0.000013
★epoch 8	loss_t: 0.000014	loss_v: 0.000012
★epoch 9	loss_t: 0.000012	loss_v: 0.000010
★epoch 10	loss_t: 0.000011	loss_v: 0.000009
★epoch 11	loss_t: 0.000010	loss_v: 0.000008
 epoch 12	loss_t: 0.000010	loss_v: 0.000009
 epoch 13	loss_t: 0.000008	loss_v: 0.000010
★epoch 14	loss_t: 0.000008	loss_v: 0.000007
 epoch 15	loss_t: 0.000008	loss_v: 0.000008
★epoch 16	loss_t: 0.000008	loss_v: 0.000006
★epoch 17	loss_t: 0.000008	loss_v: 0.000006
 epoch 18	loss_t: 0.000008	loss_v: 0.000006
 epoch 19	loss_t: 0.000007	loss_v: 0.000009
★epoch 20	loss_t: 0.000007	loss_v: 0.000005
 epoch 21	loss_t: 0.000007	loss_v: 0.000006
 epoch 22	loss_t: 0.000007	loss_v: 0.00000

In [122]:
model_best = MLP()
model_best.load_state_dict(best_model_state)
model_best(dataset_test.x)

tensor([[ 0.1649],
        [ 0.3094],
        [ 0.1052],
        ...,
        [ 0.0750],
        [ 0.2231],
        [-0.0121]], grad_fn=<AddmmBackward0>)

In [124]:
torch.save(model_best.state_dict(), 'model_released.pth')